# Patient-safe splits and GroupKFold audit

**Public role:** Core split construction and leakage-control notebook.

**Relation to the manuscript:** Supports patient-safe partitioning and no patient overlap claims.

**Execution status of this curated copy:** Stored outputs and execution counters were removed to prevent disclosure of local paths, embedded dataset images, stale errors, and large widget state. The code was not re-executed after this mechanical curation. Reference outputs are distributed separately under `results/`.

**Path configuration:** Set `AQPR_PROJECT_ROOT` to the working directory containing `project_data/`, `models/`, and `results/`. Set `AQPR_DATA_ROOT` to the external dataset root. The default values target a Google Drive/Colab workflow and can be changed without editing the notebook logic.

**Original source notebook:** `02_hygd_patient_safe_splits_and_groupkfold.ipynb`


In [ ]:
import os
# Cell 1: Load clean metadata and project paths

from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

JOURNAL_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))
CONFIG_PATH = JOURNAL_ROOT / "configs" / "project_config.json"
PATH_REGISTRY_PATH = JOURNAL_ROOT / "configs" / "path_registry.csv"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Missing config: {CONFIG_PATH}")

if not PATH_REGISTRY_PATH.exists():
    raise FileNotFoundError(f"Missing path registry: {PATH_REGISTRY_PATH}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)

path_registry = pd.read_csv(PATH_REGISTRY_PATH)
PATHS = {
    row["path_key"]: Path(row["path"])
    for _, row in path_registry.iterrows()
}

METADATA_CSV = PATHS["metadata"] / "metadata_clean.csv"

if not METADATA_CSV.exists():
    raise FileNotFoundError(f"Missing metadata_clean.csv: {METADATA_CSV}")

metadata = pd.read_csv(METADATA_CSV)

required_columns = [
    "image_id",
    "image_name",
    "patient_id",
    "label_raw",
    "label",
    "patient_label",
    "quality_score",
    "image_path",
    "image_exists",
]

missing_columns = [col for col in required_columns if col not in metadata.columns]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}. "
        f"Available columns: {list(metadata.columns)}"
    )

expected_images = int(PROJECT_CONFIG["expected_hygd_images"])
expected_patients = int(PROJECT_CONFIG["expected_hygd_patients"])
n_images = len(metadata)
n_patients = metadata["patient_id"].nunique()
mixed_patients = metadata.groupby("patient_id")["label"].nunique().gt(1).sum()

print(f"Images: {n_images}")
print(f"Patients: {n_patients}")
print(f"Mixed-label patients: {mixed_patients}")
print(f"Labels: {metadata['label_raw'].value_counts().to_dict()}")

if n_images != expected_images:
    raise ValueError(f"Expected {expected_images} images, found {n_images}")

if n_patients != expected_patients:
    raise ValueError(f"Expected {expected_patients} patients, found {n_patients}")

if mixed_patients != 0:
    raise ValueError(f"Mixed-label patients found: {mixed_patients}")

print("Status: PASS")

In [ ]:
# Cell 2: Create patient-safe train/val/test split

metadata["patient_id"] = metadata["patient_id"].astype(str)

patient_table = (
    metadata.groupby("patient_id")
    .agg(
        patient_label=("label", "max"),
        n_images=("image_id", "count"),
        quality_mean=("quality_score", "mean"),
    )
    .reset_index()
)

target_patient_counts = {
    "train": 201,
    "val": 43,
    "test": 44,
}

target_image_counts = {
    "train": 502,
    "val": 124,
    "test": 121,
}

candidate_rows = []
best_split = None
best_score = np.inf

for split_seed in range(5000):
    patients_dev, patients_test = train_test_split(
        patient_table,
        test_size=target_patient_counts["test"],
        stratify=patient_table["patient_label"],
        random_state=split_seed,
    )

    patients_train, patients_val = train_test_split(
        patients_dev,
        test_size=target_patient_counts["val"],
        stratify=patients_dev["patient_label"],
        random_state=split_seed,
    )

    split_parts = {
        "train": patients_train.copy(),
        "val": patients_val.copy(),
        "test": patients_test.copy(),
    }

    image_counts = {
        split_name: int(split_df["n_images"].sum())
        for split_name, split_df in split_parts.items()
    }

    patient_class_counts = {
        f"{split_name}_positive_patients": int(split_df["patient_label"].sum())
        for split_name, split_df in split_parts.items()
    }

    image_count_error = sum(
        abs(image_counts[split_name] - target_image_counts[split_name])
        for split_name in target_image_counts
    )

    test_image_penalty = abs(image_counts["test"] - target_image_counts["test"]) * 2
    score = image_count_error + test_image_penalty

    candidate_rows.append(
        {
            "split_seed": split_seed,
            "score": score,
            "train_images": image_counts["train"],
            "val_images": image_counts["val"],
            "test_images": image_counts["test"],
            **patient_class_counts,
        }
    )

    if score < best_score:
        best_score = score
        best_split = {
            "split_seed": split_seed,
            "parts": split_parts,
            "image_counts": image_counts,
        }

patients_train = best_split["parts"]["train"].copy()
patients_val = best_split["parts"]["val"].copy()
patients_test = best_split["parts"]["test"].copy()

patients_train["split"] = "train"
patients_val["split"] = "val"
patients_test["split"] = "test"

patient_split = pd.concat(
    [patients_train, patients_val, patients_test],
    axis=0
).sort_values("patient_id").reset_index(drop=True)

split_map = dict(zip(patient_split["patient_id"], patient_split["split"]))

metadata_split = metadata.copy()
metadata_split["split"] = metadata_split["patient_id"].map(split_map)

if metadata_split["split"].isna().any():
    missing = metadata_split.loc[metadata_split["split"].isna(), "patient_id"].unique().tolist()
    raise ValueError(f"Patients without split assignment: {missing[:20]}")

patient_overlap_checks = {
    "train_val": len(set(patients_train["patient_id"]) & set(patients_val["patient_id"])),
    "train_test": len(set(patients_train["patient_id"]) & set(patients_test["patient_id"])),
    "val_test": len(set(patients_val["patient_id"]) & set(patients_test["patient_id"])),
}

split_summary = (
    metadata_split.groupby("split")
    .agg(
        n_images=("image_id", "count"),
        n_patients=("patient_id", "nunique"),
        n_positive_images=("label", "sum"),
        n_negative_images=("label", lambda x: int((x == 0).sum())),
        quality_mean=("quality_score", "mean"),
        quality_std=("quality_score", "std"),
    )
    .reset_index()
)

patient_split_summary = (
    patient_split.groupby("split")
    .agg(
        n_patients=("patient_id", "count"),
        n_positive_patients=("patient_label", "sum"),
        n_negative_patients=("patient_label", lambda x: int((x == 0).sum())),
        images_mean=("n_images", "mean"),
        images_min=("n_images", "min"),
        images_max=("n_images", "max"),
    )
    .reset_index()
)

candidate_splits = pd.DataFrame(candidate_rows).sort_values("score").reset_index(drop=True)

patient_split_csv = PATHS["splits"] / "train_val_test_patient_split.csv"
image_split_csv = PATHS["splits"] / "train_val_test_image_split.csv"
split_summary_csv = PATHS["results_02_splits"] / "notebook02_train_val_test_split_summary.csv"
patient_split_summary_csv = PATHS["results_02_splits"] / "notebook02_patient_split_summary.csv"
candidate_splits_csv = PATHS["results_02_splits"] / "notebook02_candidate_split_search.csv"

patient_split.to_csv(patient_split_csv, index=False)
metadata_split.to_csv(image_split_csv, index=False)
split_summary.to_csv(split_summary_csv, index=False)
patient_split_summary.to_csv(patient_split_summary_csv, index=False)
candidate_splits.to_csv(candidate_splits_csv, index=False)

print(f"Selected split seed: {best_split['split_seed']}")
print(f"Patient split counts: {patient_split['split'].value_counts().to_dict()}")
print(f"Image split counts: {metadata_split['split'].value_counts().to_dict()}")
print(f"Patient overlaps: {patient_overlap_checks}")
print(f"Candidate split search: {candidate_splits_csv}")

if any(value != 0 for value in patient_overlap_checks.values()):
    raise RuntimeError(f"Patient leakage detected: {patient_overlap_checks}")

if patient_split["patient_id"].nunique() != expected_patients:
    raise ValueError("Unexpected number of split patients.")

if len(metadata_split) != expected_images:
    raise ValueError("Unexpected number of split images.")

print("Status: PASS")

In [ ]:
# Cell 3: Create repeated patient-safe GroupKFold assignments

dev_patient_split = patient_split[patient_split["split"].isin(["train", "val"])].copy()
test_patient_split = patient_split[patient_split["split"] == "test"].copy()

fold_patient_rows = []
fold_image_rows = []
fold_summary_rows = []

for seed in PROJECT_CONFIG["seeds"]:
    skf = StratifiedKFold(
        n_splits=int(PROJECT_CONFIG["n_folds"]),
        shuffle=True,
        random_state=int(seed),
    )

    X = dev_patient_split["patient_id"].values
    y = dev_patient_split["patient_label"].values

    for fold_id, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        cv_train_patients = dev_patient_split.iloc[train_idx].copy()
        cv_val_patients = dev_patient_split.iloc[val_idx].copy()

        cv_train_ids = set(cv_train_patients["patient_id"])
        cv_val_ids = set(cv_val_patients["patient_id"])

        overlap = len(cv_train_ids & cv_val_ids)
        if overlap != 0:
            raise RuntimeError(f"Patient overlap found for seed={seed}, fold={fold_id}")

        for role, fold_patients in [
            ("cv_train", cv_train_patients),
            ("cv_val", cv_val_patients),
        ]:
            temp_patients = fold_patients.copy()
            temp_patients["seed"] = seed
            temp_patients["fold"] = fold_id
            temp_patients["fold_role"] = role
            fold_patient_rows.append(temp_patients)

            temp_images = metadata_split[
                metadata_split["patient_id"].isin(fold_patients["patient_id"])
            ].copy()
            temp_images["seed"] = seed
            temp_images["fold"] = fold_id
            temp_images["fold_role"] = role
            fold_image_rows.append(temp_images)

        fold_summary_rows.append(
            {
                "seed": seed,
                "fold": fold_id,
                "train_patients": cv_train_patients["patient_id"].nunique(),
                "val_patients": cv_val_patients["patient_id"].nunique(),
                "train_images": int(cv_train_patients["n_images"].sum()),
                "val_images": int(cv_val_patients["n_images"].sum()),
                "train_positive_patients": int(cv_train_patients["patient_label"].sum()),
                "val_positive_patients": int(cv_val_patients["patient_label"].sum()),
                "patient_overlap": overlap,
            }
        )

fold_patient_assignments = pd.concat(fold_patient_rows, axis=0).reset_index(drop=True)
fold_image_assignments = pd.concat(fold_image_rows, axis=0).reset_index(drop=True)
fold_summary = pd.DataFrame(fold_summary_rows)

fold_patient_csv = PATHS["splits_groupkfold"] / "groupkfold_patient_assignments.csv"
fold_image_csv = PATHS["splits_groupkfold"] / "groupkfold_image_assignments.csv"
fold_summary_csv = PATHS["splits_groupkfold"] / "groupkfold_fold_summary.csv"

fold_patient_results_csv = PATHS["results_02_splits"] / "notebook02_groupkfold_patient_assignments.csv"
fold_image_results_csv = PATHS["results_02_splits"] / "notebook02_groupkfold_image_assignments.csv"
fold_summary_results_csv = PATHS["results_02_splits"] / "notebook02_groupkfold_fold_summary.csv"

fold_patient_assignments.to_csv(fold_patient_csv, index=False)
fold_image_assignments.to_csv(fold_image_csv, index=False)
fold_summary.to_csv(fold_summary_csv, index=False)

fold_patient_assignments.to_csv(fold_patient_results_csv, index=False)
fold_image_assignments.to_csv(fold_image_results_csv, index=False)
fold_summary.to_csv(fold_summary_results_csv, index=False)

expected_fold_rows = len(PROJECT_CONFIG["seeds"]) * int(PROJECT_CONFIG["n_folds"])
max_overlap = int(fold_summary["patient_overlap"].max())

print(f"Seeds: {PROJECT_CONFIG['seeds']}")
print(f"Folds per seed: {PROJECT_CONFIG['n_folds']}")
print(f"Fold summaries: {len(fold_summary)}")
print(f"Max patient overlap: {max_overlap}")
print(f"Patient assignments: {fold_patient_csv}")
print(f"Image assignments: {fold_image_csv}")

if len(fold_summary) != expected_fold_rows:
    raise ValueError(f"Expected {expected_fold_rows} fold summaries, found {len(fold_summary)}")

if max_overlap != 0:
    raise RuntimeError("Patient leakage detected in GroupKFold assignments.")

print("Status: PASS")

In [ ]:
# Cell 4: Split integrity audit

dev_patient_ids = set(dev_patient_split["patient_id"])
test_patient_ids = set(test_patient_split["patient_id"])

fold_test_overlap = set(fold_patient_assignments["patient_id"]) & test_patient_ids
fold_non_dev_patients = set(fold_patient_assignments["patient_id"]) - dev_patient_ids

val_coverage_rows = []

for seed in PROJECT_CONFIG["seeds"]:
    seed_val = fold_patient_assignments[
        (fold_patient_assignments["seed"] == seed) &
        (fold_patient_assignments["fold_role"] == "cv_val")
    ]

    val_counts = seed_val["patient_id"].value_counts()
    missing_val_patients = dev_patient_ids - set(val_counts.index)
    repeated_val_patients = val_counts[val_counts != 1]

    val_coverage_rows.append(
        {
            "seed": seed,
            "dev_patients": len(dev_patient_ids),
            "val_unique_patients": seed_val["patient_id"].nunique(),
            "missing_val_patients": len(missing_val_patients),
            "repeated_val_patients": len(repeated_val_patients),
            "passed": len(missing_val_patients) == 0 and len(repeated_val_patients) == 0,
        }
    )

val_coverage = pd.DataFrame(val_coverage_rows)

split_integrity = pd.DataFrame(
    [
        {
            "check": "train_val_test_patient_overlap",
            "value": max(patient_overlap_checks.values()),
            "expected": 0,
            "passed": max(patient_overlap_checks.values()) == 0,
        },
        {
            "check": "test_patients_excluded_from_cv",
            "value": len(fold_test_overlap),
            "expected": 0,
            "passed": len(fold_test_overlap) == 0,
        },
        {
            "check": "cv_patients_all_from_development_set",
            "value": len(fold_non_dev_patients),
            "expected": 0,
            "passed": len(fold_non_dev_patients) == 0,
        },
        {
            "check": "val_coverage_once_per_seed",
            "value": int(val_coverage["passed"].sum()),
            "expected": len(PROJECT_CONFIG["seeds"]),
            "passed": bool(val_coverage["passed"].all()),
        },
        {
            "check": "holdout_test_patients",
            "value": len(test_patient_ids),
            "expected": target_patient_counts["test"],
            "passed": len(test_patient_ids) == target_patient_counts["test"],
        },
        {
            "check": "holdout_test_images",
            "value": int((metadata_split["split"] == "test").sum()),
            "expected": target_image_counts["test"],
            "passed": int((metadata_split["split"] == "test").sum()) == target_image_counts["test"],
        },
    ]
)

split_integrity_csv = PATHS["results_02_splits"] / "notebook02_split_integrity_audit.csv"
val_coverage_csv = PATHS["results_02_splits"] / "notebook02_cv_val_coverage_by_seed.csv"

split_integrity.to_csv(split_integrity_csv, index=False)
val_coverage.to_csv(val_coverage_csv, index=False)

failed_checks = split_integrity.loc[~split_integrity["passed"], "check"].tolist()

print(f"Integrity checks passed: {len(split_integrity) - len(failed_checks)}/{len(split_integrity)}")
print(f"Test patients in CV: {len(fold_test_overlap)}")
print(f"CV validation coverage passed seeds: {int(val_coverage['passed'].sum())}/{len(val_coverage)}")
print(f"Integrity audit: {split_integrity_csv}")

if failed_checks:
    raise RuntimeError(f"Failed split integrity checks: {failed_checks}")

print("Status: PASS")

In [ ]:
# Cell 5: Save frozen split configuration

split_config = {
    "split_version": "hygd_patient_safe_split_v1",
    "split_search_selected_seed": int(best_split["split_seed"]),
    "target_patient_counts": target_patient_counts,
    "target_image_counts": target_image_counts,
    "actual_patient_counts": {
        split: int(count)
        for split, count in patient_split["split"].value_counts().to_dict().items()
    },
    "actual_image_counts": {
        split: int(count)
        for split, count in metadata_split["split"].value_counts().to_dict().items()
    },
    "cv_seeds": [int(seed) for seed in PROJECT_CONFIG["seeds"]],
    "n_folds": int(PROJECT_CONFIG["n_folds"]),
    "holdout_test_is_excluded_from_cv": True,
    "patient_split_csv": str(patient_split_csv),
    "image_split_csv": str(image_split_csv),
    "groupkfold_patient_assignments_csv": str(fold_patient_csv),
    "groupkfold_image_assignments_csv": str(fold_image_csv),
    "groupkfold_fold_summary_csv": str(fold_summary_csv),
}

split_config_json = PATHS["splits"] / "split_config.json"
split_config_results_json = PATHS["results_02_splits"] / "notebook02_split_config.json"

with open(split_config_json, "w", encoding="utf-8") as f:
    json.dump(split_config, f, indent=2)

with open(split_config_results_json, "w", encoding="utf-8") as f:
    json.dump(split_config, f, indent=2)

test_patient_ids_csv = PATHS["splits"] / "holdout_test_patient_ids.csv"
test_image_ids_csv = PATHS["splits"] / "holdout_test_image_ids.csv"

test_patient_split[["patient_id", "patient_label", "n_images", "quality_mean"]].to_csv(
    test_patient_ids_csv, index=False
)

metadata_split.loc[
    metadata_split["split"] == "test",
    ["image_id", "image_name", "patient_id", "label", "label_raw", "quality_score", "image_path"]
].to_csv(test_image_ids_csv, index=False)

print(f"Selected split seed: {best_split['split_seed']}")
print(f"Split config: {split_config_json}")
print(f"Hold-out patients: {test_patient_ids_csv}")
print(f"Hold-out images: {test_image_ids_csv}")

if not split_config_json.exists():
    raise FileNotFoundError(f"Split config was not saved: {split_config_json}")

if not test_patient_ids_csv.exists():
    raise FileNotFoundError(f"Hold-out patient file was not saved: {test_patient_ids_csv}")

if not test_image_ids_csv.exists():
    raise FileNotFoundError(f"Hold-out image file was not saved: {test_image_ids_csv}")

print("Status: PASS")

In [ ]:
# Cell 6: Final status

required_outputs = {
    "patient_split_csv": patient_split_csv,
    "image_split_csv": image_split_csv,
    "split_summary_csv": split_summary_csv,
    "patient_split_summary_csv": patient_split_summary_csv,
    "candidate_splits_csv": candidate_splits_csv,
    "fold_patient_csv": fold_patient_csv,
    "fold_image_csv": fold_image_csv,
    "fold_summary_csv": fold_summary_csv,
    "split_integrity_csv": split_integrity_csv,
    "val_coverage_csv": val_coverage_csv,
    "split_config_json": split_config_json,
    "test_patient_ids_csv": test_patient_ids_csv,
    "test_image_ids_csv": test_image_ids_csv,
}

output_checks = {
    key: path.exists()
    for key, path in required_outputs.items()
}

actual_patient_counts = patient_split["split"].value_counts().to_dict()
actual_image_counts = metadata_split["split"].value_counts().to_dict()

content_checks = {
    "all_images_assigned": len(metadata_split) == expected_images,
    "all_patients_assigned": patient_split["patient_id"].nunique() == expected_patients,
    "target_patient_counts": actual_patient_counts == target_patient_counts,
    "target_image_counts": actual_image_counts == target_image_counts,
    "no_train_val_test_patient_overlap": max(patient_overlap_checks.values()) == 0,
    "test_excluded_from_cv": len(fold_test_overlap) == 0,
    "cv_patients_from_dev_only": len(fold_non_dev_patients) == 0,
    "fold_summary_rows": len(fold_summary) == len(PROJECT_CONFIG["seeds"]) * int(PROJECT_CONFIG["n_folds"]),
    "cv_val_coverage": bool(val_coverage["passed"].all()),
    "no_cv_patient_overlap": int(fold_summary["patient_overlap"].max()) == 0,
}

final_checks = {**output_checks, **content_checks}
failed_checks = [key for key, passed in final_checks.items() if not passed]
status = "PASS" if not failed_checks else "FAIL"

notebook02_status = {
    "notebook": "02_hygd_patient_safe_splits_and_groupkfold",
    "status": status,
    "selected_split_seed": int(best_split["split_seed"]),
    "patient_counts": {k: int(v) for k, v in actual_patient_counts.items()},
    "image_counts": {k: int(v) for k, v in actual_image_counts.items()},
    "cv_seeds": [int(seed) for seed in PROJECT_CONFIG["seeds"]],
    "n_folds": int(PROJECT_CONFIG["n_folds"]),
    "fold_summaries": int(len(fold_summary)),
    "resnet50_compatible_split_counts": actual_image_counts == target_image_counts,
    "failed_checks": failed_checks,
    "next_notebook": "03_fundus_preprocessing_fov_quality_audit.ipynb",
}

status_json = PATHS["results_02_splits"] / "notebook02_final_status.json"
status_txt = PATHS["results_02_splits"] / "notebook02_FINAL_STATUS.txt"
status_checks_csv = PATHS["results_02_splits"] / "notebook02_final_status_checks.csv"

with open(status_json, "w", encoding="utf-8") as f:
    json.dump(notebook02_status, f, indent=2)

with open(status_txt, "w", encoding="utf-8") as f:
    for key, value in notebook02_status.items():
        f.write(f"{key}: {value}\n")

pd.DataFrame(
    [{"check": key, "passed": bool(value)} for key, value in final_checks.items()]
).to_csv(status_checks_csv, index=False)

print(f"Status: {status}")
print(f"Selected split seed: {best_split['split_seed']}")
print(f"Patient counts: {actual_patient_counts}")
print(f"Image counts: {actual_image_counts}")
print(f"Fold summaries: {len(fold_summary)}")
print(f"Final status: {status_json}")

if failed_checks:
    raise RuntimeError(f"Notebook 02 failed checks: {failed_checks}")